# BrainAI on Colab
Runtime > Change runtime type > **T4 GPU**. Everything lives inside the cloned repo on local disk (wiped when the session ends), so run the last cell to download weights.
Run top to bottom. Cells 1-4 are data; training starts at section 5.

In [ ]:
# 1) repo + dependencies
!apt-get -qq install git-lfs && git lfs install
import os
if not os.path.exists('/content/BrainAI'):
    !git clone https://github.com/ZahidServers/BrainAI.git /content/BrainAI
%cd /content/BrainAI
!git pull --ff-only
!pip -q install datasets pillow opencv-python-headless soundfile transformers accelerate pyarrow
import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Text corpora (reading / writing)
Written straight into `data/`, where the trainer auto-discovers every `.txt` as a domain. 100 MB per source is plenty: the loader samples ~30 MB from each.

In [ ]:
!python tools/fetch_text.py --out data --mb 100

## 3. Sensor data (images, audio, video)

In [ ]:
!python tools/fetch_sensors.py --out sensors --n 20000

## 4. Robot + humanoid data (LeRobot format)

In [ ]:
!python tools/fetch_robot.py --out robot_data --skip koch_pick_place_1_lego
!python tools/fetch_robot.py --out robot_data --humanoid

## 5. Smoke test the language trainer (about 1-2 minutes). It must print `device: cuda` and write `checkpoints/smoke.pt`.

In [ ]:
!python train_multidomain.py --scale small --steps 100 --batch 16 --seq 256 --eval-every 50 --out checkpoints/smoke.pt --history checkpoints/smoke_history.json
!ls -la checkpoints

## 6. Language training (multi-domain). Saves every 500 steps and on any crash or stop, so weights always exist.
To resume after a disconnect that kept the files, add `--init checkpoints/brainai_multidomain.pt`.

In [ ]:
!python train_multidomain.py --scale small --steps 40000 --batch 32 --seq 256 --save-every 500 --eval-every 1000

## 7. Perception encoders (held-out accuracy printed against chance)

In [ ]:
for m, s, b in [('images', 6000, 64), ('audio', 3000, 32), ('video', 1500, 16)]:
    !python train_sensors.py --modality $m --steps $s --batch $b --scale small

## 8. Behaviour cloning (state -> action) on every robot / humanoid dataset

In [ ]:
import glob
for d in sorted(glob.glob('robot_data/*')):
    !python train_robot.py --dataset $d --steps 3000 --scale small

## 9. (Optional) Distil Qwen: the teacher writes text, BrainAI learns from it. Slow; start small.

In [ ]:
!python distill.py generate --teacher Qwen/Qwen2.5-1.5B-Instruct --samples 3000

## 10. Download the weights (local disk is wiped when the session ends)

In [ ]:
!zip -qr brainai_checkpoints.zip checkpoints
from google.colab import files
files.download('brainai_checkpoints.zip')